# Implementing a Simple Neural Network on the Titanic Dataset

**Goal:** Build a neural network (using TensorFlow/Keras) to predict whether a Titanic passenger survived, based on passenger features such as class, sex, age, and fare.


## 1. Introduction

The Titanic dataset is one of the most well-known datasets in data science and machine learning. It contains details of passengers aboard the RMS Titanic and their survival status. This notebook builds a simple neural network model to predict whether a passenger survived, based on the available features.

## 2. Problem Statement

The objective is to classify passengers as either survivors or non-survivors using a neural network. We will preprocess the data, train a neural network, and evaluate its performance using standard classification metrics.


In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_curve, auc,
                              ConfusionMatrixDisplay)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)



: 

In [ ]:
import sys
print(sys.executable)

## 3. Dataset Information

* **Source:** https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

**Key Features**

| Column | Description |
|---|---|
| PassengerId | Unique ID of the passenger |
| Pclass | Ticket class: 1 = First, 2 = Second, 3 = Third |
| Name | Passenger's name |
| Sex | Male/Female |
| Age | Age in years |
| SibSp | Number of siblings/spouses aboard |
| Parch | Number of parents/children aboard |
| Ticket | Ticket number |
| Fare | Ticket fare |
| Cabin | Cabin number |
| Embarked | Port of embarkation: C = Cherbourg, Q = Queenstown, S = Southampton |
| Survived | Target variable: 0 = No, 1 = Yes |


In [ ]:
# Load the dataset
url =r"C:\Users\anupa\OneDrive\Desktop\titanic.xlsx"
df = pd.read_excel(url)

print("Shape:", df.shape)
df.head()


In [ ]:
# Quick look at data types and missing values
df.info()


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nSurvival rate:")
print(df['Survived'].value_counts(normalize=True))


## 4. Data Preprocessing

We'll go through the standard steps in order:

1. **Handling missing values** — impute `Age` and `Embarked`, drop `Cabin` (too many missing values).
2. **Dropping irrelevant columns** — `PassengerId`, `Name`, `Ticket` don't carry predictive signal in raw form.
3. **Encoding categorical data** — convert `Sex` and `Embarked` to numeric.
4. **Feature scaling** — standardize numeric features so the network trains well.
5. **Splitting data** — into train and test sets.


In [ ]:
# 4.1 Handling missing values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Cabin has too many missing values to be useful directly, so we drop it
df = df.drop(columns=['Cabin'])

# 4.2 Drop columns that aren't useful as raw numeric/categorical features
df = df.drop(columns=['PassengerId', 'Name', 'Ticket'])

print(df.isnull().sum())


In [ ]:
# 4.3 Encoding categorical data
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

df.head()


In [ ]:
# 4.4 Split features/target, then train/test split
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4.5 Feature scaling (fit on train only, apply to both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)


## 5. Model Implementation

### Architecture
* **Input layer:** one node per feature
* **Hidden layers:** Dense layers with ReLU activation
* **Output layer:** single node with sigmoid activation (binary classification)

### Compilation
* **Loss:** binary cross-entropy
* **Optimizer:** Adam
* **Metric:** accuracy


In [ ]:
# Define the neural network architecture
n_features = X_train_scaled.shape[1]

model = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
# Train the model, monitoring validation performance to avoid overfitting
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss over epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy over epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Model Evaluation

We assess performance on the held-out test set using accuracy, precision, recall, F1-score, a confusion matrix, and the ROC curve.


In [ ]:
# Predictions on the test set
y_pred_proba = model.predict(X_test_scaled).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Did not survive', 'Survived'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


## 7. Conclusion

**Key findings** *(fill in with your actual numbers after running the notebook)*:
- The model achieved an accuracy of **~X%**, precision of **~X**, recall of **~X**, and an F1-score of **~X** on the test set.
- The ROC-AUC of **~X** indicates the model's ability to separate survivors from non-survivors.
- Sex and passenger class tend to be strong predictors of survival on this dataset (women and first-class passengers had notably higher survival rates).

**Potential improvements:**
- **Hyperparameter tuning** — try different numbers of layers/neurons, learning rates, batch sizes, and dropout rates (e.g., via `keras_tuner` or grid search).
- **Feature engineering** — extract titles from `Name` (Mr., Mrs., Miss, etc.), create a `FamilySize` feature from `SibSp` + `Parch`, or bin `Age`/`Fare`.
- **Handling class imbalance** — use class weights if survival classes are imbalanced.
- **More complex architectures** — deeper networks, batch normalization, or ensembling with tree-based models (Random Forest, XGBoost) for comparison.
- **Cross-validation** — use k-fold cross-validation instead of a single train/test split for a more robust performance estimate.
